In [ ]:
# Step 1: Generate 2D Head Dataset and Velocity Dataset
import numpy as np
import os
import pandas as pd
import h5py
from numba import jit, prange

# Parameter settings
params = {
    'p': 3, 'q': 3, 'HRx': 6, 'HLx': 3, 'LX': 10000, 'D': -2000, 'a': 2/3, 'b': 1/3, 'TauD': 1
}
Kwater =1 # hydraulic conductivity K=1m/d

# Extract parameters
p, q, HRx, HLx, LX, D, a, b, TauD = params['p'], params['q'], params['HRx'], params['HLx'], params['LX'], params['D'], params['a'], params['b'], params['TauD']

# Define the mesh grid 
mesh_x, mesh_z = 201, 201
x_range = np.linspace(0, LX, mesh_x)
z_range = np.linspace(D, 0, mesh_z)

# Precompute cosine values
b1 = np.cos(np.pi * x_range / LX)
b2 = np.cos(p * np.pi * x_range / LX)

@jit(nopython=True, parallel=True)
def compute_J(mesh_x, mesh_z, x_range, z_range, HRx, HLx, LX, D, a, b, TauD, b1, b2, c1, c2, e1, e2):
    """Calculate the J matrix and related variables in 2D (x, z) space."""
    J = np.zeros((mesh_x, mesh_z))
    for i in prange(mesh_x):
        for k in prange(mesh_z):
            # Calculate potential components hA, hB
            hA = HRx - HRx * b1[i] * np.cosh(np.pi * (z_range[k] - D) / LX) / c1
            hB = HLx - HLx * b2[i] * np.cosh(p * np.pi * (z_range[k] - D) / LX) / c2
            g1, g2 = 0.0, 0.0
            for n in range(1, 1000):  
                k1, an = (-1) ** (n - 1), (2 * n - 1) * np.pi / 2
                b0 = np.cos(an * (z_range[k] - D) / D)
                f1 = D * (k1 * an * b0 * (1 / an**2 + (an**2 * e1 + 2 * np.pi * TauD * e2) / (an**4 + 4 * np.pi**2 * TauD**2)))
                g1 += f1
                k2, d1 = (-1) ** n, an**2 + (p * np.pi * D / LX)**2
                f2 = D * (k2 * an * b0 * (1 / d1 + (d1 * e1 + 2 * np.pi * TauD * e2) / (d1**2 + 4 * np.pi**2 * TauD**2)))
                g2 += f2               
            hC = HLx * g1 / D + HLx * b2[i] * g2 / D
            J[i, k] = hA + a * hB + b * hC
    return J

def calculate_fields(tp):
    """Calculate the potential field (J), velocity components (vx, vz), and velocity magnitude (UU) for a given time point (tp)."""
    e1 = np.cos(2 * np.pi * tp)
    e2 = np.sin(2 * np.pi * tp)

    # Precompute cosh values for efficiency
    c1 = np.cosh(np.pi * D / LX)
    c2 = np.cosh(p * np.pi * D / LX)

    J = compute_J(mesh_x, mesh_z, x_range, z_range, HRx, HLx, LX, D, a, b, TauD, b1, b2, c1, c2, e1, e2)

    # Compute the gradient of J (only x and z directions)
    dx, dz = x_range[1] - x_range[0], z_range[1] - z_range[0]
    grad_x, grad_z = np.gradient(J, dx, dz)

    # Calculate the velocity components as the negative gradient of J
    vx, vz = -1 * Kwater * grad_x, -1 * Kwater * grad_z

    # Calculate the magnitude of the velocity field
    UU = np.sqrt(vx**2 + vz**2)

    # Hydraulic head (H) is equivalent to J
    H = J

    return H, vx, vz, UU

def save_data_hdf5(filename, J, vx, vz, UU, H, x_range, z_range):
    """Save the computed data into an HDF5 file."""
    with h5py.File(filename, 'w') as f:
        f.create_dataset('J', data=J)
        f.create_dataset('vx', data=vx)
        f.create_dataset('vz', data=vz)
        f.create_dataset('UU', data=UU)
        f.create_dataset('H', data=H)
        f.create_dataset('x_range', data=x_range)
        f.create_dataset('z_range', data=z_range)

def save_data_tecplot(filename, vx, vz, UU, J, x_range, z_range):
    """Save the computed data into a Tecplot-compatible .dat file."""
    # Apply mask for z_range < -20
    mask = z_range < -20

    x, z = np.meshgrid(x_range, z_range, indexing='ij')
    
    # Flatten and filter data
    x = x[:, mask].flatten()
    z = z[:, mask].flatten()
    vx = vx[:, mask].flatten()
    vz = vz[:, mask].flatten()
    UU = UU[:, mask].flatten()
    J = J[:, mask].flatten()

    # Stack data column-wise
    data = np.column_stack((x, z, vx, vz, UU, J))

    # Sort data by x, then z
    data = data[np.lexsort((x, z))]

    with open(filename, 'w') as f:
        f.write('TITLE = "Velocity, Speed and Head Data"\n')
        f.write('VARIABLES = "X", "Z", "VX", "VZ", "UU", "H"\n')
        f.write(f'ZONE T="Flow Field", I={len(x_range)}, K={len(z_range[mask])}, F=POINT\n')
        np.savetxt(f, data, fmt='%.6f', delimiter=' ')

# Setup output directory
current_working_dir = os.getcwd()
result_folder = os.path.join(current_working_dir, 'results')
os.makedirs(result_folder, exist_ok=True)

# Create a file to store UU statistics
stats_filename = os.path.join(result_folder, 'UU_statistics.txt')
with open(stats_filename, 'w') as stats_file:
    stats_file.write("TimePoint,UU_Max,UU_Min,UU_25thPercentile\n")

# Loop over time points (tp) and compute, save results
for tp in np.arange(0, 1, 0.05):
    H, vx, vz, UU = calculate_fields(tp)

    # Calculate UU statistics
    uu_max = np.max(UU)
    uu_min = np.min(UU)
    uu_25th_percentile = np.percentile(UU, 25)

    # Save UU statistics to the file
    with open(stats_filename, 'a') as stats_file:
        stats_file.write(f"{tp:.2f},{uu_max:.6f},{uu_min:.6f},{uu_25th_percentile:.6f}\n")

    # Construct file names for the current time point
    hdf5_filename = os.path.join(result_folder, f'head_and_velocity_tp_{tp:.2f}.h5')
    tecplot_filename = os.path.join(result_folder, f'head_and_velocity_tp_{tp:.2f}.dat')
    
    # Save data to HDF5 format
    save_data_hdf5(hdf5_filename, H, vx, vz, UU, H, x_range, z_range)
    
    # Save data to Tecplot .dat format
    save_data_tecplot(tecplot_filename, vx, vz, UU, H, x_range, z_range)

    # Print progress
    print(f"Completed processing for tp = {tp:.2f}")
    print(f"UU Statistics: Max={uu_max:.6f}, Min={uu_min:.6f}, 25th Percentile={uu_25th_percentile:.6f}")

# End of processing
print("All time points processed and data saved.")

In [ ]:
# Step 2: Finding Out Stagnation Points in 2D profile
import os
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import plotly.graph_objects as go
import plotly.io as pio
from scipy.interpolate import griddata
from scipy.ndimage import minimum_filter
from concurrent.futures import ThreadPoolExecutor

# Get the current working directory
script_dir = Path.cwd()

# Define the range boundaries
x_min, x_max = 0, 10000
z_min, z_max = -2000, -40
threshold = 0.001

# Define a function to filter points that are at least 200 meters apart
def filter_points(points):
    filtered = []
    for point in points:
        if all(np.linalg.norm(np.array(point) - np.array(p)) >= 200 for p in filtered):
            filtered.append(point)
    return filtered

def process_xz_domain(X_filtered, Z_filtered, vx_filtered, vz_filtered, UU_filtered):
    # Create a grid for interpolation
    grid_x, grid_z = np.mgrid[x_min:x_max:800j, z_min:z_max:800j]  # Define a finer grid for interpolation

    # Perform interpolation for vx, vz, UU using griddata
    grid_vx = griddata((X_filtered, Z_filtered), vx_filtered, (grid_x, grid_z), method='linear')
    grid_vz = griddata((X_filtered, Z_filtered), vz_filtered, (grid_x, grid_z), method='linear')
    grid_UU = griddata((X_filtered, Z_filtered), UU_filtered, (grid_x, grid_z), method='linear')

    # Identify local minima in UU using a minimum filter
    UU_local_min = (grid_UU == minimum_filter(grid_UU, size=50))  # Find local minima in UU
    UU_local_min_points = []

    # Collect points that meet the threshold condition (UU = 0 or very close to 0)
    for i in range(grid_x.shape[0]):
        for j in range(grid_x.shape[1]):
            if UU_local_min[i, j] and x_min + 1 <= grid_x[i, j] <= x_max - 1 and z_min + 1 <= grid_z[i, j] <= z_max - 1:
                if grid_UU[i, j] < threshold:
                    UU_local_min_points.append((grid_x[i, j], grid_z[i, j], grid_UU[i, j]))

    return filter_points(UU_local_min_points)

# Aggregate and filter points from all slices
def compare_and_filter_points(points):
    filtered = []
    for point in points:
        existing_point = next((p for p in filtered if np.linalg.norm(np.array(point[:2]) - np.array(p[:2])) < 1), None)
        if existing_point:
            if point[2] < existing_point[2]:
                filtered.remove(existing_point)
                filtered.append(point)
        else:
            filtered.append(point)
    return filtered

# Save points to a CSV file
def save_points_to_csv(points, file_name, header):
    csv_path = script_dir / 'results' / file_name
    np.savetxt(csv_path, points, delimiter=', ', header=header, comments='', fmt='%s')
    print(f"Found and saved {len(points)} points to {file_name}")

# Process a CSV file and generate a Tecplot .dat file
def process_csv_file(file_prefix, title="Stagnation Points"):
    """
    Process a CSV file and generate a Tecplot .dat file.

    :param file_prefix: The prefix of the CSV file (without the .csv extension).
    :param title: The title for the ZONE in the Tecplot .dat file.
    """
    input_path = script_dir / 'results' / f'{file_prefix}.csv'
    df = pd.read_csv(input_path)

    # Remove any spaces in column names
    df.columns = df.columns.str.strip()

    # Ensure required columns are present
    required_columns = ['x', 'z']
    if 'tp' in df.columns:
        required_columns.append('tp')
    actual_columns = df.columns.tolist()

    if not all(col in actual_columns for col in required_columns):
        print(f"Error: CSV file is missing required columns. Actual columns: {actual_columns}")
        return

    # Save the .dat file with required formatting
    output_path_dat = script_dir / 'results' / f'{file_prefix}.dat'
    with open(output_path_dat, 'w') as f:
        # Write the VARIABLES declaration
        if 'tp' in df.columns:
            f.write('VARIABLES = "tp", "X", "Z", "UU"\n')
        else:
            f.write('VARIABLES = "X", "Z", "UU"\n')

        # Write the ZONE header
        num_points = len(df)
        f.write(f'ZONE T="{title}", I={num_points}, J=1, K=1, F=POINT\n')

        # Write the data points
        for index, row in df.iterrows():
            if 'tp' in df.columns:
                f.write(f"{row['tp']}\t{row['x']}\t{row['z']}\t{row['UU']}\n")
            else:
                f.write(f"{row['x']}\t{row['z']}\t{row['UU']}\n")

    print(f'File generated: {output_path_dat}')

# Generate a 2D scatter plot for visualization
def plot_points(all_points, title):
    fig = go.Figure()
    base_colors = ['#FF0000', '#FFA500', '#008000', '#87CEEB', '#00FFFF', '#0000FF', '#FF00CC', '#A52A2A', '#808080', '#000000']
    
    # Extend the color list if necessary
    num_colors_needed = len(all_points)
    colors = (base_colors * (num_colors_needed // len(base_colors) + 1))[:num_colors_needed]

    for tp, color in zip(sorted(all_points.keys()), colors):
        points = all_points[tp]
        if len(points) == 0:  # Check if points list is empty
            print(f"No points found for tp={tp:.2f}, skipping...")
            continue  # Skip this iteration if no points are found
        x, z, _ = zip(*points)
        fig.add_trace(go.Scatter(
            x=x,
            y=z,
            mode='markers',
            marker=dict(size=5, color=color),
            name=f'tp={tp}'
        ))

    # Update the layout of the figure
    fig.update_layout(
        title=title,
        xaxis_title='X',
        yaxis_title='Z',
        xaxis=dict(range=[0, 10000]),  # Set x-axis range from 4000 to 6000
        yaxis=dict(range=[-2000, 0])
        #template="plotly_white"
    )

    return fig

# Main function where data is loaded and processing is done
def main():
    all_points = {}
    all_stagnation_points = []  # List to collect all stagnation points across all time steps

    for tp in np.arange(0, 1, 0.05):
        file_relative_path = Path('results', f'head_and_velocity_tp_{tp:.2f}.h5')
        file_path = script_dir / file_relative_path

        # Check if the HDF5 file exists
        if not file_path.exists():
            print(f"Error: File '{file_path}' does not exist!")
            continue

        try:
            # Load data from the HDF5 file
            with h5py.File(file_path, 'r') as f:
                x_range = f['x_range'][:]
                z_range = f['z_range'][:]
                vx = f['vx'][:]
                vz = f['vz'][:]
                UU = f['UU'][:]
        except Exception as e:
            print(f"Error loading file: {e}")
            continue

        # Apply the filtering criteria to extract relevant data
        X, Z = np.meshgrid(x_range, z_range, indexing='ij')
        filtered_indices = (
            (X >= x_min) & (X <= x_max) &
            (Z >= z_min) & (Z <= z_max)
        )
        X_filtered = X[filtered_indices]
        Z_filtered = Z[filtered_indices]
        vx_filtered = vx[filtered_indices]
        vz_filtered = vz[filtered_indices]
        UU_filtered = UU[filtered_indices]

        # Process the entire xz domain for the given time step
        UU_local_min_points = process_xz_domain(X_filtered, Z_filtered, vx_filtered, vz_filtered, UU_filtered)

        # Filter and save the local minima points for each time step (tp)
        UU_local_min_points_filtered = compare_and_filter_points(UU_local_min_points)
        save_points_to_csv(UU_local_min_points_filtered, f'best_xz_tp_{tp:.2f}.csv', 'x, z, UU\n')
        process_csv_file(f'best_xz_tp_{tp:.2f}', title=f'Stagnation Points at tp={tp:.2f}')
        all_points[tp] = UU_local_min_points_filtered

        # Collect all stagnation points for this time step
        for point in UU_local_min_points_filtered:
            all_stagnation_points.append((tp, *point))  # Add the time point (tp) to the point tuple

    # Save all stagnation points to a single CSV file
    all_stagnation_points_df = pd.DataFrame(all_stagnation_points, columns=['tp', 'x', 'z', 'UU'])
    all_stagnation_csv_path = script_dir / 'results' / 'all_stagnation_points.csv'
    all_stagnation_points_df.to_csv(all_stagnation_csv_path, index=False)
    print(f"Saved all stagnation points to {all_stagnation_csv_path}")

    # Convert the CSV file to a Tecplot .dat file
    process_csv_file('all_stagnation_points', title='All Stagnation Points')

    # Plot the points for all time steps
    fig_best_xz = plot_points(all_points, 'Best Points from XZ Domain with Local Min UU')
    output_html_file = script_dir / 'results' / 'Stagline_Case1T_xz.html'
    pio.write_html(fig_best_xz, file=output_html_file, auto_open=False)
    fig_best_xz.show()

if __name__ == '__main__':
    main()

In [ ]:
# Step 3: Determining critical points based on Hessian Matrix method
import os
import numpy as np
import pandas as pd
import logging
from pathlib import Path
import plotly.graph_objects as go
from scipy.linalg import eig
import h5py

# Set up logging to track events during processing
logging.basicConfig(filename='processing.log', level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Function to read the CSV file containing stagnation points (x, z)
def read_best_points(file_prefix):
    """
    Reads the CSV file containing the stagnation points (x, z coordinates).

    Args:
        file_prefix (str): Prefix of the CSV file to read.

    Returns:
        pd.DataFrame: DataFrame containing the stagnation points, or None if the file is empty or does not exist.
    """
    input_path = Path('results') / f'{file_prefix}.csv'
    if not input_path.exists() or input_path.stat().st_size == 0:
        logging.warning(f"File {input_path} does not exist or is empty.")
        return None
    
    try:
        df = pd.read_csv(input_path)
        df.columns = df.columns.str.strip()  # Strip spaces from column names
        return df
    except (pd.errors.EmptyDataError, pd.errors.ParserError) as e:
        logging.error(f"Error reading {input_path}: {e}")
        return None

# Function to compute the Hessian matrix at a point
def compute_hessian(H, x, z, x_range, z_range, epsilon=1e-3):
    """
    Computes the Hessian matrix of the potential field at a given point (x, z).

    Args:
        H (np.ndarray): 2D array of the potential field.
        x (float): x-coordinate of the point.
        z (float): z-coordinate of the point.
        x_range (np.ndarray): Array of x-coordinates.
        z_range (np.ndarray): Array of z-coordinates.
        epsilon (float): Small perturbation for finite differences.

    Returns:
        np.ndarray: Hessian matrix at the point (x, z).
    """
    # Find the closest x and z values in the range arrays
    idx_x = np.abs(x_range - x).argmin()
    idx_z = np.abs(z_range - z).argmin()

    # Extract the potential field values around the point
    phi_x_plus_epsilon = H[idx_x + 1, idx_z]
    phi_x_minus_epsilon = H[idx_x - 1, idx_z]
    phi_z_plus_epsilon = H[idx_x, idx_z + 1]
    phi_z_minus_epsilon = H[idx_x, idx_z - 1]
    phi_x_plus_z_plus_epsilon = H[idx_x + 1, idx_z + 1]
    phi_x_plus_z_minus_epsilon = H[idx_x + 1, idx_z - 1]
    phi_x_minus_z_plus_epsilon = H[idx_x - 1, idx_z + 1]
    phi_x_minus_z_minus_epsilon = H[idx_x - 1, idx_z - 1]

    # Second derivatives using central differences
    d2phi_dx2 = (phi_x_plus_epsilon - 2 * H[idx_x, idx_z] + phi_x_minus_epsilon) / (epsilon ** 2)
    d2phi_dz2 = (phi_z_plus_epsilon - 2 * H[idx_x, idx_z] + phi_z_minus_epsilon) / (epsilon ** 2)
    d2phi_dxdz = (phi_x_plus_z_plus_epsilon - phi_x_plus_z_minus_epsilon -
                  phi_x_minus_z_plus_epsilon + phi_x_minus_z_minus_epsilon) / (4 * epsilon ** 2)
    
    # Hessian matrix
    hessian = np.array([[d2phi_dx2, d2phi_dxdz],
                        [d2phi_dxdz, d2phi_dz2]])
    return hessian

# Function to find starting points for separating streamlines using Hessian matrix
def find_starting_points(H, stagnation_points, x_range, z_range, epsilon=500):
    """
    Finds starting points for separating streamlines using the Hessian matrix.

    Args:
        H (np.ndarray): 2D array of the potential field.
        stagnation_points (list): List of stagnation points (x, z).
        x_range (np.ndarray): Array of x-coordinates.
        z_range (np.ndarray): Array of z-coordinates.
        epsilon (float): Larger perturbation for finite differences to increase point separation.

    Returns:
        dict: Dictionary of starting points for inflow and outflow directions.
    """
    starting_points = {'inflow1': [], 'inflow2': [], 'outflow1': [], 'outflow2': []}
    
    for point in stagnation_points:
        x, z = point
        hessian = compute_hessian(H, x, z, x_range, z_range, epsilon)
        
        # Compute eigenvalues and eigenvectors
        eigenvalues, eigenvectors = eig(hessian)
        
        # Sort eigenvalues and eigenvectors by magnitude of eigenvalues
        sorted_indices = np.argsort(np.abs(eigenvalues))
        eigenvalues = eigenvalues[sorted_indices]
        eigenvectors = eigenvectors[:, sorted_indices]
        
        # Determine directions based on eigenvalues
        for i, eigenvalue in enumerate(eigenvalues):
            direction = eigenvectors[:, i]
            direction = direction / np.linalg.norm(direction)  # Normalize the direction vector
            
            if eigenvalue > 0:  # Positive curvature (flow towards the stagnation point)
                starting_points['inflow1'].append([x + direction[0] * epsilon, z + direction[1] * epsilon])
                starting_points['inflow2'].append([x - direction[0] * epsilon, z - direction[1] * epsilon])
            elif eigenvalue < 0:  # Negative curvature (flow away from the stagnation point)
                starting_points['outflow1'].append([x + direction[0] * epsilon, z + direction[1] * epsilon])
                starting_points['outflow2'].append([x - direction[0] * epsilon, z - direction[1] * epsilon])
    
    return starting_points

# Function to save tracking points to a CSV file
def save_tracking_points(file_name, tracking_points):
    """
    Saves the tracking points to a CSV file.

    Args:
        file_name (str): Name of the CSV file to save.
        tracking_points (dict): Dictionary of tracking points.
    """
    output_path = Path('results') / file_name
    with open(output_path, 'w') as f:
        f.write("direction,x,z\n")
        for key, points in tracking_points.items():
            for point in points:
                f.write(f"{key},{point[0]},{point[1]}\n")
    logging.info(f"Saved tracking points to {output_path}")

# Function to plot points and tracking points
def plot_points(stagnation_points, tracking_points, file_name, tp):
    """
    Plots the stagnation points and tracking points.

    Args:
        stagnation_points (np.ndarray): Array of stagnation points (x, z).
        tracking_points (dict): Dictionary of tracking points.
        file_name (str): Name of the HTML file to save.
        tp (float): Time point.
    """
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=stagnation_points[:, 0],
        y=stagnation_points[:, 1],
        mode='markers',
        marker=dict(size=10, color='red'),
        name='Stagnation Points'
    ))

    colors = {'inflow1': 'blue', 'inflow2': 'cyan', 'outflow1': 'green', 'outflow2': 'lime'}
    for key, points in tracking_points.items():
        if not points:  # Check if the list is empty
            logging.warning(f"No tracking points found for type '{key}' at tp={tp:.2f}.")
            continue

        points = np.array(points)
        if points.ndim == 1:  # Check if the array is one-dimensional
            logging.warning(f"Tracking points for type '{key}' at tp={tp:.2f} are one-dimensional. Skipping.")
            continue

        fig.add_trace(go.Scatter(
            x=points[:, 0],
            y=points[:, 1],
            mode='markers',
            marker=dict(size=6, color=colors[key]),
            name=f'{key.capitalize()} Points'
        ))

    fig.update_layout(
        title=f'Stagnation and Tracking Points at tp={tp:.2f}',
        xaxis_title='X',
        yaxis_title='Z',
        xaxis=dict(range=[0, 10000]),
        yaxis=dict(range=[-2000, 0])
    )

    output_path = Path('results') / file_name
    fig.write_html(output_path)
    logging.info(f"Generated plot {output_path}")
    
# Main function to process all files and generate outputs
def main():
    """
    Main function to process all files and generate tracking points, plots, and output files.
    """
    result_folder = Path('results')
    result_folder.mkdir(exist_ok=True)  # Ensure the results directory exists

    logging.info("Starting processing of tp values.")
    
    # Define the mesh grid
    mesh_x, mesh_z = 101, 101
    x_range = np.linspace(0, 10000, mesh_x)
    z_range = np.linspace(-2000, 0, mesh_z)

    for tp in np.arange(0, 1, 0.05):
        file_prefix = f'best_xz_tp_{tp:.2f}'
        stagnation_points_df = read_best_points(file_prefix)
        if stagnation_points_df is None:
            continue
        
        stagnation_points = stagnation_points_df[['x', 'z']].values
        
        # Load the water head data from the HDF5 file for the current time point
        hdf5_filename = result_folder / f'head_and_velocity_tp_{tp:.2f}.h5'
        if not hdf5_filename.exists():
            logging.warning(f"HDF5 file {hdf5_filename} does not exist.")
            continue
        
        with h5py.File(hdf5_filename, 'r') as f:
            H = f['H'][:]
        
        # Generate tracking points using Hessian matrix with increased epsilon
        tracking_points = find_starting_points(H, stagnation_points, x_range, z_range, epsilon=50)
        
        save_tracking_points(f'{file_prefix}_tracking_points.csv', tracking_points)
        plot_points(stagnation_points, tracking_points, f'{file_prefix}_tracking_points.html', tp)
    
    logging.info("Processing completed successfully.")
    print("Code execution completed successfully.")

if __name__ == '__main__':
    main()

In [ ]:
import numpy as np
import h5py
import os
import pandas as pd
import plotly.graph_objects as go
from scipy.integrate import solve_ivp
from concurrent.futures import ThreadPoolExecutor
from numba import jit
import plotly.io as pio
import logging

# Set up logging to track events during processing
logging.basicConfig(filename='processing.log', level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Function to read data from an HDF5 file (only x and z components of velocity)
def read_data_hdf5(filename):
    with h5py.File(filename, 'r') as f:
        vx = f['vx'][:]  # Velocity component in x-direction
        vz = f['vz'][:]  # Velocity component in z-direction
        x_range = f['x_range'][:]  # Range of x values
        z_range = f['z_range'][:]  # Range of z values
    return vx, vz, x_range, z_range

# JIT-compiled function to compute the velocity field at a given position
@jit(nopython=True)
def velocity_field_numba(pos, vx, vz, x_range, z_range):
    x, z = pos
    # Check if the position is outside the defined ranges
    if x < x_range[0] or x > x_range[-1] or z < z_range[0] or z > z_range[-1]:
        return np.array([0.0, 0.0])  # Return zero velocity if out of bounds
    
    # Find the indices of the grid cell containing the position
    xi = np.searchsorted(x_range, x) - 1
    zi = np.searchsorted(z_range, z) - 1

    # Get the surrounding grid points
    x1, x2 = x_range[xi], x_range[xi+1]
    z1, z2 = z_range[zi], z_range[zi+1]

    # Compute relative distances within the grid cell
    xd = (x - x1) / (x2 - x1)
    zd = (z - z1) / (z2 - z1)

    # Trilinear interpolation for the x-component of velocity
    vx_val = vx[xi, zi] * (1 - xd) + vx[xi + 1, zi] * xd

    # Trilinear interpolation for the z-component of velocity
    vz_val = vz[xi, zi] * (1 - xd) + vz[xi + 1, zi] * xd

    return np.array([vx_val, vz_val])  # Return the interpolated velocity vector

# Function to compute streamlines based on velocity fields and start points
def compute_streamlines(vx, vz, x_range, z_range, start_points, max_distance=1e7, tol=1e-7):
    # Define the velocity field function for the ODE solver
    def velocity_field(t, pos):
        return velocity_field_numba(pos, vx, vz, x_range, z_range)

    # Define an event to terminate integration when a boundary condition is met
    def boundary_event(t, pos):
        return pos[1] + 60  # Termination condition, e.g., z = -60 surface

    boundary_event.terminal = True  # Stop integration when event is triggered
    boundary_event.direction = 0     # Event is detected regardless of direction

    # Function to integrate the streamline starting from a given point
    def integrate_streamline(start):
        if start[1] > -60:  # Skip if the start point is above the boundary
            return np.array([])

        # Integrate forward in time
        result_forward = solve_ivp(velocity_field, [0, max_distance], start, method='RK45', rtol=tol, atol=tol, events=boundary_event)
        
        # Integrate backward in time
        result_backward = solve_ivp(velocity_field, [0, -max_distance], start, method='RK45', rtol=tol, atol=tol, events=boundary_event)
        
        # Extract the streamline points
        streamline_forward = result_forward.y.T
        streamline_backward = result_backward.y.T
        
        # Reverse the backward streamline and combine with forward streamline
        streamline_backward = streamline_backward[::-1]
        streamline = np.vstack((streamline_backward, streamline_forward))
        
        return streamline

    # Use a thread pool to parallelize streamline computations
    with ThreadPoolExecutor() as executor:
        streamlines = list(executor.map(integrate_streamline, start_points))
    
    return streamlines  # Return the list of computed streamlines

# Function to plot streamlines and their corresponding start points
def plot_streamlines_and_points(fig, streamlines, start_points, directions, colors):
    for streamline, start, direction in zip(streamlines, start_points, directions):
        if len(streamline) > 0:
            # Ensure the direction is a valid key in the colors dictionary
            if direction in colors:
                color = colors[direction]
            else:
                logging.warning(f"Direction '{direction}' not found in colors list. Using default color.")
                color = 'black'  # Default color if direction is not found

            fig.add_trace(go.Scatter(
                x=streamline[:, 0], y=streamline[:, 1],
                mode='lines',
                line=dict(color=color, width=2),  # Set streamline color and width
                name=f'Streamline {direction}'
            ))
    
    # Plot the start points of the streamlines
    for start, direction in zip(start_points, directions):
        if direction in colors:
            color = colors[direction]
        else:
            logging.warning(f"Direction '{direction}' not found in colors list. Using default color.")
            color = 'black'  # Default color if direction is not found

        fig.add_trace(go.Scatter(
            x=[start[0]], y=[start[1]],
            mode='markers',
            marker=dict(color=color, size=5, symbol='square'),  # Set marker color, size, and shape
            name=f'Start Point {direction}'
        ))

# Function to read trace points from a CSV file
def read_trace_points(file_path):
    # Read the CSV file
    df = pd.read_csv(file_path, names=['direction', 'x', 'z'], header=0)
    
    directions = df['direction'].astype(str).tolist()
    start_points = df[['x', 'z']].values  # Return only the x, z coordinates
    
    return directions, start_points

# Get the current working directory as the base path for the script
current_working_dir = os.getcwd()

# Define the path to the results folder and create it if it doesn't exist
result_folder = os.path.join(current_working_dir, 'results')
os.makedirs(result_folder, exist_ok=True)

# Define a list of colors to be used for different tracking point categories
colors = {
    'inflow1': 'magenta',
    'inflow2': 'green',
    'outflow1': 'red',
    'outflow2': 'blue'
}

# Define prefixes for the tracking point files
tracking_point_prefixes = [
    'best_xz'
]

# Loop through different tp values to process each corresponding file
for tp in np.arange(0, 1, 0.25):   # Iterate over tp from 0 to 1 with step 0.25
    # Define the path to the input HDF5 file for the current tp value
    input_path_hdf5 = os.path.join(result_folder, f'head_and_velocity_tp_{tp:.2f}.h5')
    
    # Read data from the HDF5 file
    vx, vz, x_range, z_range = read_data_hdf5(input_path_hdf5)

    # Create a new Plotly figure for visualization
    fig = go.Figure()

    # Loop through each tracking point category and its corresponding color
    for prefix in tracking_point_prefixes:
        # Define the path to the tracking point CSV file
        file_path = os.path.join(result_folder, f'{prefix}_tp_{tp:.2f}_tracking_points.csv')
        # Read the start points and directions for streamlines
        directions, start_points = read_trace_points(file_path)
        
        # Compute streamlines based on the velocity field and start points
        streamlines = compute_streamlines(vx, vz, x_range, z_range, start_points)
        
        # Plot the computed streamlines and their start points on the figure
        plot_streamlines_and_points(fig, streamlines, start_points, directions, colors)

    # Update the layout of the plot to set axis ranges
    fig.update_layout(
        xaxis=dict(title='X', range=[x_range[0], x_range[-1]]),
        yaxis=dict(title='Z', range=[z_range[0], z_range[-1]],scaleanchor="x", scaleratio=1),
        title=f'Streamlines at tp = {tp:.2f}',
        showlegend=True
    )

    # Save the plot as an HTML file and display it
    html_file_path = os.path.join(result_folder, f'streamlines_tp_{tp:.2f}.html')
    fig.write_html(html_file_path)
    fig.show()  # Show the plot in the browser